# ChurnIQ: Customer Churn Prediction & Retention Intelligence

**DevOps & AI Mini Project**  
**Author:** Barath Kumar V  
**Model:** XGBoost Classifier with SHAP Explainability  
**Dataset:** IBM Telco Customer Churn (7,043 accounts)  
**Stack:** FastAPI REST API + React TypeScript Dashboard  


## 1. Setup & Environment
Install required packages for data science, gradient boosting, explainable AI, and API serving.

In [1]:
# !pip install xgboost shap fastapi uvicorn scikit-learn pandas numpy
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, roc_curve
print('All libraries imported successfully.')


## 2. Data Loading & Cleaning
Load the Telco Churn dataset and clean non-numeric records.

In [2]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
print(f'Dataset shape: {df.shape}')

# Impute missing TotalCharges with MonthlyCharges * tenure
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'])
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df.head()


## 3. Feature Engineering
Extract business domain features:
- **Contract Risk**: Month-to-month contracts
- **Fiber Risk**: Fiber optic internet churn risk
- **M2M Fiber Risk**: Month-to-month + Fiber optic interaction
- **Long Tenure Safe**: Customers with >24 months tenure
- **Charge Per Tenure**: Monthly cost burden relative to loyalty

In [3]:
df['ContractRisk'] = (df['Contract'] == 'Month-to-month').astype(int)
df['FiberRisk'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['M2MFiberRisk'] = ((df['Contract'] == 'Month-to-month') & (df['InternetService'] == 'Fiber optic')).astype(int)
df['LongTenureSafe'] = (df['tenure'] > 24).astype(int)
df['ChargePerTenure'] = df['MonthlyCharges'] / (df['tenure'] + 1.0)
df['NoSupport'] = (df['TechSupport'] == 'No').astype(int)
df['NoSecurity'] = (df['OnlineSecurity'] == 'No').astype(int)
print('Feature engineering complete.')


## 4. Stratified Model Training (XGBoost)
Split 80/20 with stratification and fit XGBoost classifier.

In [4]:
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ContractRisk', 'FiberRisk', 'M2MFiberRisk', 'LongTenureSafe', 'ChargePerTenure', 'NoSupport', 'NoSecurity']
X = df[features]
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(f'AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')


## 5. SHAP Explainability
Compute TreeExplainer SHAP values to explain reasons for churn.

In [5]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
ranking = pd.DataFrame({'Feature': features, 'MeanAbsSHAP': mean_abs_shap}).sort_values('MeanAbsSHAP', ascending=False)
print(ranking.to_string(index=False))


## 6. Run FastAPI Service
Serve predictions to the ChurnIQ React dashboard.

In [6]:
# Launch FastAPI server:
# uvicorn main:app --host 0.0.0.0 --port 8000
